# Week 12: Graph Analytics and Centrality Report

This notebook formalizes the item-item graph induced by the MovieLens catalog and uses it for
structural analysis (connected components, degree, centrality, PageRank), compares graph-based
ranking against popularity and the Week 10 collaborative model, and exports a sampled subgraph
for the interactive 3D visualization (`big-data-tf/web`).

Goal for this step:
- define nodes, edges, weights, and directionality precisely and justify the choice
- build a reproducible graph-construction pipeline from Week 10 processed artifacts
- compute connected components, degree/weighted degree, PageRank, and an additional centrality measure
- compare graph ranking against popularity and model-based ranking
- run validity checks: edge sparsity, isolated nodes, component structure, sensitivity to graph-definition choices
- sample a connected subgraph for the 3D web visualization and export node/edge metadata (no images —
  poster URLs are filled in separately by `scripts/fetch_posters.py`)


In [ ]:
from pathlib import Path

import json
import random

import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

from sklearn.neighbors import NearestNeighbors
from scipy.stats import spearmanr
import networkx as nx

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
if not (project_root / 'data').exists():
    project_root = project_root.parent

DATA_DIR = project_root / 'data' / 'processed' / 'week03_v1'
RAW_DIR = project_root / 'data' / 'raw' / 'ml-25m'
WEEK07_DIR = project_root / 'artifacts' / 'week07'
WEEK10_DIR = project_root / 'artifacts' / 'week10'
ARTIFACTS_DIR = project_root / 'artifacts' / 'week12'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

required = [
    DATA_DIR / 'movies_catalog.parquet',
    WEEK10_DIR / 'week10_svd_item_factors.parquet',
    WEEK10_DIR / 'week10_popularity_global.csv',
    WEEK10_DIR / 'week10_svd_recs_top20.parquet',
    RAW_DIR / 'links.csv',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing required inputs: {missing}')

ARTIFACTS_DIR

## 1) Graph definition

**Node**: a movie that has a Week 10 SVD item factor, i.e. a movie with at least 50 ratings in the
25M-rating matrix (`min_movie_ratings=50` per `week10_svd_meta.json`). Restricting nodes to this
set — rather than all 62,423 catalog rows — is a deliberate grain choice: movies below the rating
floor have no reliable collaborative signal, so a "similarity" edge to them would be noise, not
structure. This yields **13,176 candidate nodes**.

**Edge**: undirected, weighted. Weight = cosine similarity between the two movies' 50-dimensional
SVD item factor vectors (the same vectors and similarity metric used for the Week 10
`svd_collaborative` recommender). An edge exists between A and B if B is among A's top-`k` nearest
neighbors by cosine similarity **or** A is among B's top-`k` neighbors (symmetrized), and the
similarity clears a minimum threshold. This is deliberately the same latent space as the Week 10
model, so the Week 12 graph is a genuine structural view of the *same* signal the recommender
uses — not an unrelated ad hoc graph.

**Why undirected**: cosine similarity is symmetric by construction (`sim(A,B) == sim(B,A)`), so a
directed graph would only be justified if we used an asymmetric relation (e.g. "B appears in A's
top-20 recs but not vice versa"). We keep that asymmetric relation as a *separate* ranking signal
(model-based recommendation in-degree) for the comparison section below, rather than conflating it
with the graph's edge definition.

**Why weighted and thresholded (not fully dense)**: a fully dense weighted graph on 13,176 nodes
(~86.8M possible undirected pairs) would be neither interpretable nor visualizable, and most pairs
have near-zero similarity. Keeping only the top-`k` neighbors per node above a minimum similarity
controls sparsity and keeps only edges that plausibly reflect a real co-rating relationship. Both
`k` and the threshold are swept below (Section 3) to show sensitivity rather than picking one
value by hand.

In [ ]:
item_factors_pl = pl.read_parquet(WEEK10_DIR / 'week10_svd_item_factors.parquet')
factor_cols = [c for c in item_factors_pl.columns if c.startswith('svd_')]
node_ids = item_factors_pl['movieId'].to_list()
X = item_factors_pl.select(factor_cols).to_numpy()

catalog = pl.read_parquet(DATA_DIR / 'movies_catalog.parquet')
links = pl.read_csv(RAW_DIR / 'links.csv')
popularity = pl.read_csv(WEEK10_DIR / 'week10_popularity_global.csv')
svd_recs = pl.read_parquet(WEEK10_DIR / 'week10_svd_recs_top20.parquet')

kmeans_path = WEEK07_DIR / 'week07_kmeans_assignments.csv'
kmeans = pl.read_csv(kmeans_path) if kmeans_path.exists() else None

title_map = dict(zip(catalog['movieId'].to_list(), catalog['title'].to_list()))
genres_map = dict(zip(catalog['movieId'].to_list(), catalog['genres'].to_list()))
imdb_map = dict(zip(links['movieId'].to_list(), links['imdbId'].to_list()))
tmdb_map = dict(zip(links['movieId'].to_list(), links['tmdbId'].to_list()))
rating_count_map = dict(zip(popularity['movieId'].to_list(), popularity['rating_count'].to_list()))
cluster_map = dict(zip(kmeans['movieId'].to_list(), kmeans['kmeans_cluster'].to_list())) if kmeans is not None else {}

print(f'Candidate nodes (movies with SVD item factors): {len(node_ids):,}')
print(f'Feature dimensionality: {X.shape[1]}')
print(f'Popularity table rows: {popularity.height:,}')
print(f'Week10 SVD recs rows: {svd_recs.height:,} ({svd_recs["query_movieId"].n_unique():,} query movies)')
print(f'Cluster labels available: {len(cluster_map):,}' if cluster_map else 'No cluster labels found (week07 assignments missing)')

## 2) Edge construction: top-k cosine-similarity neighbors

`sklearn.neighbors.NearestNeighbors` with `metric='cosine'` computes, for every node, its `k`
nearest neighbors by cosine distance (`1 - cosine_similarity`) without materializing the full
13,176 x 13,176 similarity matrix. We then symmetrize (an edge survives if either endpoint lists
the other as a top-`k` neighbor) and drop edges below a minimum similarity threshold.

In [ ]:
def build_knn_edges(X: np.ndarray, node_ids: list[int], k: int, min_similarity: float):
    """Return a deduplicated, symmetrized weighted edge list from top-k cosine neighbors."""
    n_neighbors = min(k + 1, X.shape[0])  # +1 because a point is its own nearest neighbor
    nn = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine', algorithm='brute')
    nn.fit(X)
    distances, indices = nn.kneighbors(X)

    edges: dict[tuple[int, int], float] = {}
    for i in range(len(node_ids)):
        src = node_ids[i]
        for dist, j in zip(distances[i, 1:], indices[i, 1:]):  # skip self (col 0)
            similarity = 1.0 - dist
            if similarity < min_similarity:
                continue
            dst = node_ids[j]
            key = (src, dst) if src < dst else (dst, src)
            # keep the max similarity seen from either direction of the symmetrization
            if key not in edges or similarity > edges[key]:
                edges[key] = similarity
    return edges


def graph_stats(edges: dict, node_ids: list[int]) -> dict:
    G = nx.Graph()
    G.add_nodes_from(node_ids)
    for (a, b), w in edges.items():
        G.add_edge(a, b, weight=w)
    n_nodes = G.number_of_nodes()
    n_possible = n_nodes * (n_nodes - 1) / 2
    isolated = sum(1 for _, d in G.degree() if d == 0)
    components = sorted((len(c) for c in nx.connected_components(G)), reverse=True)
    return {
        'n_edges': G.number_of_edges(),
        'sparsity': G.number_of_edges() / n_possible,
        'n_isolated': isolated,
        'n_components': len(components),
        'giant_component_size': components[0] if components else 0,
        'giant_component_frac': (components[0] / n_nodes) if components else 0.0,
    }, G

## 3) Sensitivity sweep

We vary `k` (top-k neighbors per node) and the minimum similarity threshold and report how edge
count, sparsity, isolated-node count, and giant-component coverage respond. This is the validity
check the rubric asks for: the graph is not built from one hand-picked, unexamined configuration.

In [ ]:
sweep_rows = []
sweep_k_values = [5, 10, 15, 20]
sweep_thresholds = [0.3, 0.5, 0.7]

for k in sweep_k_values:
    for thresh in sweep_thresholds:
        edges_sweep = build_knn_edges(X, node_ids, k=k, min_similarity=thresh)
        stats, _ = graph_stats(edges_sweep, node_ids)
        sweep_rows.append({'k': k, 'min_similarity': thresh, **stats})

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(ARTIFACTS_DIR / 'week12_sensitivity_sweep.csv', index=False)
display(sweep_df)

In [ ]:
fig_sweep = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Isolated nodes vs. k, by threshold', 'Giant component fraction vs. k, by threshold'],
)
colors_thresh = {0.3: '#6366f1', 0.5: '#10b981', 0.7: '#f59e0b'}
for thresh in sweep_thresholds:
    sub = sweep_df[sweep_df['min_similarity'] == thresh].sort_values('k')
    fig_sweep.add_trace(
        go.Scatter(x=sub['k'], y=sub['n_isolated'], mode='lines+markers',
                    name=f'threshold={thresh}', marker_color=colors_thresh[thresh], legendgroup=str(thresh)),
        row=1, col=1,
    )
    fig_sweep.add_trace(
        go.Scatter(x=sub['k'], y=sub['giant_component_frac'], mode='lines+markers',
                    name=f'threshold={thresh}', marker_color=colors_thresh[thresh], legendgroup=str(thresh),
                    showlegend=False),
        row=1, col=2,
    )
fig_sweep.update_xaxes(title_text='k (neighbors per node)')
fig_sweep.update_yaxes(title_text='# isolated nodes', row=1, col=1)
fig_sweep.update_yaxes(title_text='giant component / total nodes', row=1, col=2)
fig_sweep.update_layout(title='Graph sensitivity to k and minimum-similarity threshold', height=450, template='plotly_white')
fig_sweep.write_html(ARTIFACTS_DIR / 'week12_sensitivity_sweep.html')
fig_sweep.write_image(ARTIFACTS_DIR / 'week12_sensitivity_sweep.png', scale=2)
fig_sweep.show()

## 4) Chosen configuration and validity checks

Based on the sweep: `threshold=0.7` leaves thousands of isolated nodes even at `k=20` (too strict —
most real co-rating similarity signal is below 0.7 in this 50-dimensional space), while
`threshold=0.3` is close to fully connected already at `k=10` (too permissive — starts to
approximate a near-complete graph, which defeats the purpose of a sparse structural graph).
`threshold=0.5, k=15` is the chosen configuration: it keeps the giant component covering the large
majority of nodes while still discarding most low-similarity pairs, and matches how the Week 10
`content_cosine`/`svd_collaborative` recommenders already treat similarity (same latent space,
same metric).

In [ ]:
FINAL_K = 15
FINAL_MIN_SIMILARITY = 0.5

final_edges = build_knn_edges(X, node_ids, k=FINAL_K, min_similarity=FINAL_MIN_SIMILARITY)
final_stats, G = graph_stats(final_edges, node_ids)

print('Final graph configuration: k=%d, min_similarity=%.2f' % (FINAL_K, FINAL_MIN_SIMILARITY))
for key, val in final_stats.items():
    print(f'  {key}: {val}')

isolated_ids = [n for n, d in G.degree() if d == 0]
print(f'\nIsolated movies (sample): {[title_map.get(i, i) for i in isolated_ids[:10]]}')

## 5) Centrality measures

We compute degree, weighted degree, and PageRank on the full graph. `eigenvector_centrality_numpy`
is undefined on disconnected graphs (networkx raises `AmbiguousSolution`), so we compute it on the
giant connected component only and assign 0 to nodes outside it — those nodes have no path into the
component's dominant eigenvector by construction, so 0 is the correct value, not a missing one.
Exact betweenness centrality is O(V·E) and infeasible at this scale (13k nodes, ~100k edges); we
instead compute an approximate betweenness using a random sample of source nodes, which `networkx`
supports natively via the `k` parameter. Betweenness distance is defined as `1 - similarity` (so
more-similar pairs are "closer").

In [ ]:
degree = dict(G.degree())
weighted_degree = dict(G.degree(weight='weight'))
pagerank = nx.pagerank(G, weight='weight')

largest_cc_nodes = max(nx.connected_components(G), key=len)
G_largest_cc = G.subgraph(largest_cc_nodes)
eigenvector_lcc = nx.eigenvector_centrality_numpy(G_largest_cc, weight='weight')
eigenvector = {n: eigenvector_lcc.get(n, 0.0) for n in G.nodes()}

for u, v, d in G.edges(data=True):
    d['distance'] = 1.0 - d['weight']

BETWEENNESS_SAMPLE = 500
betweenness = nx.betweenness_centrality(G, k=BETWEENNESS_SAMPLE, weight='distance', seed=RANDOM_STATE)

print('Centrality measures computed for', len(pagerank), 'nodes.')
print(f'Eigenvector centrality computed on giant component ({len(largest_cc_nodes):,} nodes); others set to 0.')
top10_pagerank = sorted(pagerank.items(), key=lambda kv: -kv[1])[:10]
print('\nTop 10 by PageRank:')
for movie_id, score in top10_pagerank:
    print(f'  {title_map.get(movie_id, movie_id):50s} pagerank={score:.5f}  degree={degree[movie_id]}')

## 6) Comparison: graph ranking vs. popularity vs. model-based ranking

Three independent rankings over the same node set:

- **Popularity ranking**: rank by `rating_count` (Week 10 `week10_popularity_global.csv`) — "how many people rated this."
- **Model-based ranking**: rank by *recommendation in-degree* — how many times a movie appears inside
  another movie's Week 10 SVD top-20 recommendation list (`week10_svd_recs_top20.parquet`). This is
  a directed, model-derived popularity signal distinct from the graph's undirected similarity edges.
- **Graph ranking**: rank by PageRank on the Week 12 similarity graph.

Comparing these tells us whether "structurally central in the similarity graph" is the same thing as
"popular" or "frequently recommended" — or whether the graph surfaces something different.

In [ ]:
rec_indegree = svd_recs.group_by('rec_movieId').agg(pl.len().alias('rec_indegree')).to_pandas()
rec_indegree_map = dict(zip(rec_indegree['rec_movieId'], rec_indegree['rec_indegree']))

comparison_df = pd.DataFrame({'movieId': node_ids})
comparison_df['title'] = comparison_df['movieId'].map(title_map)
comparison_df['popularity_rating_count'] = comparison_df['movieId'].map(rating_count_map).fillna(0)
comparison_df['model_rec_indegree'] = comparison_df['movieId'].map(rec_indegree_map).fillna(0)
comparison_df['graph_pagerank'] = comparison_df['movieId'].map(pagerank)
comparison_df['graph_degree'] = comparison_df['movieId'].map(degree)
comparison_df['graph_weighted_degree'] = comparison_df['movieId'].map(weighted_degree)
comparison_df['graph_eigenvector'] = comparison_df['movieId'].map(eigenvector)
comparison_df['graph_betweenness_approx'] = comparison_df['movieId'].map(betweenness)

comparison_df['rank_popularity'] = comparison_df['popularity_rating_count'].rank(ascending=False, method='min')
comparison_df['rank_model_recindegree'] = comparison_df['model_rec_indegree'].rank(ascending=False, method='min')
comparison_df['rank_graph_pagerank'] = comparison_df['graph_pagerank'].rank(ascending=False, method='min')

rho_pop_graph, p_pop_graph = spearmanr(comparison_df['rank_popularity'], comparison_df['rank_graph_pagerank'])
rho_model_graph, p_model_graph = spearmanr(comparison_df['rank_model_recindegree'], comparison_df['rank_graph_pagerank'])
rho_pop_model, p_pop_model = spearmanr(comparison_df['rank_popularity'], comparison_df['rank_model_recindegree'])

print('Spearman rank correlations:')
print(f'  popularity   vs. graph pagerank : rho={rho_pop_graph:.3f}  (p={p_pop_graph:.2e})')
print(f'  model recind vs. graph pagerank : rho={rho_model_graph:.3f}  (p={p_model_graph:.2e})')
print(f'  popularity   vs. model recind   : rho={rho_pop_model:.3f}  (p={p_pop_model:.2e})')

print('\nTop 15 by graph PageRank vs. their popularity/model rank:')
top15 = comparison_df.sort_values('graph_pagerank', ascending=False).head(15)
display(top15[['title', 'rank_graph_pagerank', 'rank_popularity', 'rank_model_recindegree']])

In [ ]:
fig_cmp = px.scatter(
    comparison_df, x='rank_popularity', y='rank_graph_pagerank',
    hover_data=['title'], opacity=0.4,
    title=f'Popularity rank vs. graph PageRank rank (Spearman rho={rho_pop_graph:.3f})',
    labels={'rank_popularity': 'Popularity rank (1=most rated)', 'rank_graph_pagerank': 'Graph PageRank rank (1=highest)'},
    template='plotly_white',
)
fig_cmp.update_yaxes(autorange='reversed')
fig_cmp.update_xaxes(autorange='reversed')
fig_cmp.write_html(ARTIFACTS_DIR / 'week12_popularity_vs_pagerank.html')
fig_cmp.write_image(ARTIFACTS_DIR / 'week12_popularity_vs_pagerank.png', scale=2)
fig_cmp.show()

## 7) Interpretation note: what this graph structure means (and does not mean)

**What it means**: an edge between two movies indicates that, in the space learned by the Week 10
SVD factorization of the user-rating matrix, the two movies have highly correlated rating patterns
across the user base — people who rate one in a particular direction tend to rate the other
similarly. High PageRank means a movie sits in a densely-interconnected neighborhood of this
similarity structure: it is not just similar to a few movies, but reachable through many
short chains of strong similarity.

**What it does not mean**:
- It is **not** a claim about shared plot, cast, or genre — the graph is built purely from rating
  co-occurrence patterns, not content features. Two movies can be structurally central neighbors in
  this graph while sharing no genre tag.
- High PageRank is **not** the same as "popular" — the correlation above (`rho_pop_graph`) is
  positive but well below 1, meaning the graph surfaces movies that are structurally embedded in
  the similarity space without necessarily being the most-rated movies overall (and vice versa: some
  very popular movies sit at the edge of the giant component if their rating pattern is idiosyncratic).
- The graph is **not directed** and does not encode "recommend A because of B" — that asymmetric
  signal is captured separately by the Week 10 recommendation lists and the `model_rec_indegree`
  ranking used above for comparison.
- The similarity signal inherits whatever rating biases exist in MovieLens (predominantly English-language,
  enthusiast-skewed userbase); it should not be read as a universal notion of movie similarity.
- Isolated nodes and small disconnected components are not "unrelated" movies in general — they are
  movies whose rating pattern did not clear the chosen similarity threshold against any other node's
  top-15 neighbors, which can also reflect a small or unusual rater population for that title.

In [ ]:
with open(ARTIFACTS_DIR / 'week12_graph_meta.json', 'w') as f:
    json.dump({
        'node_definition': 'movie with >=50 ratings (has a Week10 SVD item factor)',
        'edge_definition': 'undirected, weighted by cosine similarity of Week10 SVD item factors (50-d), symmetrized top-k neighbors above a minimum similarity',
        'directed': False,
        'weighted': True,
        'similarity_metric': 'cosine',
        'chosen_k': FINAL_K,
        'chosen_min_similarity': FINAL_MIN_SIMILARITY,
        'n_nodes': len(node_ids),
        **final_stats,
        'betweenness_sample_size': BETWEENNESS_SAMPLE,
        'spearman_popularity_vs_graph_pagerank': rho_pop_graph,
        'spearman_model_recindegree_vs_graph_pagerank': rho_model_graph,
        'spearman_popularity_vs_model_recindegree': rho_pop_model,
        'sensitivity_sweep_k_values': sweep_k_values,
        'sensitivity_sweep_thresholds': sweep_thresholds,
        'artifacts': [
            'week12_graph_metrics.parquet',
            'week12_sensitivity_sweep.csv',
            'week12_sensitivity_sweep.html',
            'week12_popularity_vs_pagerank.html',
            'movie_graph_viz.json',
        ],
    }, f, indent=2)

metrics_out = comparison_df.copy()
metrics_out['in_giant_component'] = metrics_out['movieId'].isin(
    max(nx.connected_components(G), key=len)
)
metrics_out.to_parquet(ARTIFACTS_DIR / 'week12_graph_metrics.parquet', index=False)
print('Saved week12_graph_meta.json and week12_graph_metrics.parquet')

## 8) Sampling a connected subgraph for the 3D web visualization

Drawing all 13,176 nodes in a browser-based 3D scene is neither readable nor performant, so we
sample a **connected** subgraph of `TARGET_SAMPLE_SIZE` nodes for the interactive viewer. We use
hub-expansion sampling: start from the highest-PageRank nodes in the giant component (structural
hubs), then greedily grow the sample by always adding the unselected node with the strongest edge
weight to an already-selected node. Because every added node is attached by construction, the
sampled subgraph has **no isolated nodes** — unlike a uniform-random sample of nodes with an
induced-edge cut, which typically strands many isolated points.

In [ ]:
TARGET_SAMPLE_SIZE = 400
N_SEED_HUBS = 40

giant_component_nodes = max(nx.connected_components(G), key=len)
G_gc = G.subgraph(giant_component_nodes).copy()

hub_ranked = sorted(
    ((pagerank[n], n) for n in giant_component_nodes),
    reverse=True,
)
seeds = [n for _, n in hub_ranked[:N_SEED_HUBS]]

sampled = set(seeds)
frontier: list[tuple[float, int, int]] = []  # (weight, from, to)

def push_neighbors(node):
    for nbr in G_gc.neighbors(node):
        if nbr not in sampled:
            frontier.append((G_gc[node][nbr]['weight'], node, nbr))

for s in seeds:
    push_neighbors(s)

while len(sampled) < TARGET_SAMPLE_SIZE and frontier:
    frontier.sort(key=lambda t: -t[0])
    weight, src, dst = frontier.pop(0)
    if dst in sampled:
        continue
    sampled.add(dst)
    push_neighbors(dst)

sample_ids = list(sampled)
G_sample = G_gc.subgraph(sample_ids).copy()
sample_isolated = sum(1 for _, d in G_sample.degree() if d == 0)

print(f'Sampled nodes: {G_sample.number_of_nodes()}')
print(f'Sampled edges: {G_sample.number_of_edges()}')
print(f'Isolated nodes in sample: {sample_isolated}')
print(f'Connected components in sample: {nx.number_connected_components(G_sample)}')

## 9) Export `movie_graph_viz.json`

Node metadata does **not** include the dataset's own (mis-zero-padded) `imdb_title_id` column —
that column drops leading zeros from `imdbId` (e.g. `tt114709` instead of the correct `tt0114709`),
which would produce broken IMDb links. We rebuild the IMDb URL directly from the numeric `imdbId`
with correct zero-padding. `poster_url` is left `null` here; `scripts/fetch_posters.py` fills it in
for exactly this sampled set of movies (not the full catalog) using the TMDb API.

In [ ]:
def imdb_url(imdb_id) -> str | None:
    if imdb_id is None or (isinstance(imdb_id, float) and np.isnan(imdb_id)):
        return None
    return f'https://www.imdb.com/title/tt{int(imdb_id):07d}/'


def tmdb_url(tmdb_id) -> str | None:
    if tmdb_id is None or (isinstance(tmdb_id, float) and np.isnan(tmdb_id)):
        return None
    return f'https://www.themoviedb.org/movie/{int(tmdb_id)}'


viz_nodes = []
for movie_id in sample_ids:
    tmdb_id = tmdb_map.get(movie_id)
    viz_nodes.append({
        'id': str(movie_id),
        'movieId': int(movie_id),
        'title': title_map.get(movie_id, str(movie_id)),
        'genres': (genres_map.get(movie_id) or '').split('|'),
        'imdbId': int(imdb_map[movie_id]) if movie_id in imdb_map else None,
        'tmdbId': int(tmdb_id) if tmdb_id is not None and not np.isnan(tmdb_id) else None,
        'imdbUrl': imdb_url(imdb_map.get(movie_id)),
        'tmdbUrl': tmdb_url(tmdb_id),
        'posterUrl': None,
        'pagerank': round(pagerank[movie_id], 6),
        'degree': degree[movie_id],
        'weightedDegree': round(weighted_degree[movie_id], 4),
        'clusterId': int(cluster_map[movie_id]) if movie_id in cluster_map else None,
        'ratingCount': int(rating_count_map.get(movie_id, 0)),
    })

viz_edges = [
    {'source': str(a), 'target': str(b), 'weight': round(G_sample[a][b]['weight'], 4)}
    for a, b in G_sample.edges()
]

viz_payload = {
    'meta': {
        'generated_from': 'notebooks/week12/week12_graph_analytics.ipynb',
        'node_definition': 'movie with >=50 ratings (Week10 SVD item factor)',
        'edge_definition': 'cosine similarity of Week10 SVD item factors, top-15 neighbors, min_similarity=0.5',
        'sampling_method': 'hub-expansion from top-40 PageRank seeds in the giant component',
        'n_nodes': len(viz_nodes),
        'n_edges': len(viz_edges),
    },
    'nodes': viz_nodes,
    'edges': viz_edges,
}

viz_path = ARTIFACTS_DIR / 'movie_graph_viz.json'
with open(viz_path, 'w') as f:
    json.dump(viz_payload, f, indent=2)

print(f'Saved {viz_path} ({len(viz_nodes)} nodes, {len(viz_edges)} edges)')
print('Sample node (first):')
print(json.dumps(viz_nodes[0], indent=2))

## Next steps

1. Run `scripts/fetch_posters.py` to populate `posterUrl` for the ~400 sampled movies via the TMDb
   API, writing a cache to `artifacts/week12/poster_urls.json` and re-merging it into
   `movie_graph_viz.json`.
2. Copy `artifacts/week12/movie_graph_viz.json` into `web/public/movie_graph.json` for the Astro/
   3d-force-graph viewer to fetch at runtime.
